# CS 281 Milestone Experiments — Is Chain-of-Thought Just Rationalization?

Mechanistic faithfulness audit of Llama 3 8B via TransformerLens.

**Plan of execution (proposal §):**
1. Infrastructure: TransformerLens + Llama 3 8B on Colab; logit lens validated on held-out examples.
2. Logit lens results: commitment-point analysis across 500 BoolQ examples with layer × token heatmaps.
3. Preliminary patching: random CoT corruption pipeline on 200 MNLI examples.

Requires a Hugging Face token with access to `meta-llama/Meta-Llama-3-8B-Instruct`. Set it in the environment as `HF_TOKEN` or paste into the cell below.

## 1. Setup

In [ ]:
# Colab only: clone repo or upload the cot_faithfulness/ package alongside this notebook.
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'transformer-lens>=2.0', 'datasets>=2.18', 'huggingface-hub>=0.22', 'accelerate>=0.28', 'seaborn'], check=True)
sys.path.insert(0, os.path.abspath('.'))

In [ ]:
import os
HF_TOKEN = os.environ.get('HF_TOKEN') or ''  # paste here if running ad hoc
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

In [ ]:
import json, pickle, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from cot_faithfulness import data, model as model_mod, prompts, generation, logit_lens, patching, analysis

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
torch.set_grad_enabled(False)
sns.set_context('notebook'); sns.set_style('whitegrid')

## 2. Load Llama 3 8B Instruct

In [ ]:
model = model_mod.load_model()
device = next(model.parameters()).device
print(f'Loaded {model.cfg.model_name} on {device} | n_layers={model.cfg.n_layers} | d_vocab={model.cfg.d_vocab}')

In [ ]:
boolq_targets = prompts.boolq_target_tokens(model)
mnli_targets = prompts.mnli_target_tokens(model)
print('BoolQ target token ids:', boolq_targets)
print('MNLI target token ids:', mnli_targets)

## 3. Load data
500 BoolQ examples for the logit-lens sweep, 200 MNLI examples for the preliminary patching pipeline.

In [ ]:
boolq = data.load_boolq(n=500, seed=42)
mnli = data.load_mnli(n=200, seed=42)
print('BoolQ:', len(boolq), '| MNLI:', len(mnli))
boolq[0]

## 4. Sanity check: logit lens on a held-out BoolQ example
Run the logit lens on one example to verify the pipeline before scaling. We project the residual stream at every layer to the vocabulary and read out P(Yes) and P(No) at the final prompt position (the position the model is about to extend with the CoT).

In [ ]:
ex = data.boolq_example(boolq[0])
prompt = prompts.format_boolq(ex['passage'], ex['question'])
tokens = model.to_tokens(prompt)
target_ids = [boolq_targets['No'], boolq_targets['Yes']]
probs = logit_lens.logit_lens(model, tokens, target_ids, positions=[tokens.shape[1] - 1])
print('shape:', probs.shape, '(layers, positions, classes)')
print('per-layer P(No), P(Yes) at last prompt token:')
print(probs[:, 0, :])

In [ ]:
fig, ax = plt.subplots(figsize=(4, 8))
analysis.plot_layer_heatmap(probs[:, 0, :], target_names=['No', 'Yes'],
                            title=f"Q: {ex['question'][:60]}...\nlabel={ex['label_text']}", ax=ax)
plt.tight_layout(); plt.show()

## 5. Commitment-point analysis on 500 BoolQ
For each example, we record:
- per-layer P(Yes), P(No) at the last prompt token (i.e., the position right before the model would generate the CoT)
- the commitment layer: earliest layer at which P(correct class) exceeds τ = 0.8
- whether the argmax across the final layer agrees with the ground-truth label

In [ ]:
TAU = 0.8
records = []
all_probs = []
for i in tqdm(range(len(boolq))):
    ex = data.boolq_example(boolq[i])
    prompt = prompts.format_boolq(ex['passage'], ex['question'])
    tokens = model.to_tokens(prompt)
    if tokens.shape[1] > 1024:
        continue
    target_ids = [boolq_targets['No'], boolq_targets['Yes']]
    probs = logit_lens.logit_lens(model, tokens, target_ids, positions=[tokens.shape[1] - 1])[:, 0, :]
    target_idx = ex['label']
    cl = logit_lens.commitment_layer(probs, target_idx, threshold=TAU)
    pred = int(probs[-1].argmax().item())
    records.append({
        'idx': i,
        'label': ex['label'],
        'pred': pred,
        'correct': pred == ex['label'],
        'commitment_layer': cl,
        'final_p_correct': probs[-1, target_idx].item(),
    })
    all_probs.append(probs.numpy())
    if (i + 1) % 50 == 0:
        model_mod.free_memory()

all_probs = np.stack(all_probs)
np.save(RESULTS / 'boolq_layer_probs.npy', all_probs)
pd.DataFrame(records).to_csv(RESULTS / 'boolq_commitment.csv', index=False)
len(records), all_probs.shape

In [ ]:
summary = analysis.commitment_summary(records)
print(json.dumps(summary, indent=2, default=float))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
analysis.plot_commitment_histogram(records, n_layers=model.cfg.n_layers, ax=ax)
plt.tight_layout(); plt.savefig(RESULTS / 'boolq_commitment_hist.png', dpi=150); plt.show()

In [ ]:
mean_curve = all_probs.mean(axis=0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mean_curve[:, 0], label='P(No)')
ax.plot(mean_curve[:, 1], label='P(Yes)')
ax.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='τ=0.8')
ax.set_xlabel('layer'); ax.set_ylabel('logit-lens probability')
ax.set_title('Mean answer-class probability across layers (BoolQ, pre-CoT position)')
ax.legend(); plt.tight_layout(); plt.savefig(RESULTS / 'boolq_mean_curve.png', dpi=150); plt.show()

## 6. Preliminary CoT corruption on 200 MNLI
Pipeline:
1. Generate a CoT + answer from the model on each premise/hypothesis.
2. Identify the CoT token span between the prompt end and the parsed answer.
3. Replace those tokens with random vocabulary tokens.
4. Re-run the corrupted sequence and compare the answer distribution to the clean run.

A model that *uses* the CoT computationally should flip its answer when the CoT is replaced with noise. A model that *rationalizes* should keep the same answer.

In [ ]:
mnli_choices = ['entailment', 'neutral', 'contradiction']
target_ids_mnli = [mnli_targets[c] for c in mnli_choices]
patch_records = []
MAX_LEN = 1024
for i in tqdm(range(len(mnli))):
    ex = data.mnli_example(mnli[i])
    prompt = prompts.format_mnli(ex['premise'], ex['hypothesis'])
    prompt_tokens = model.to_tokens(prompt)
    if prompt_tokens.shape[1] > MAX_LEN - 180:
        continue
    completion, full_tokens = generation.generate_cot(model, prompt, max_new_tokens=160)
    parsed = generation.parse_answer(completion, mnli_choices)
    if parsed is None:
        continue
    cot_start = prompt_tokens.shape[1]
    cot_end = full_tokens.shape[1]
    corrupt_fn = lambda x, seed=i: patching.corrupt_random(model, x, seed=seed)
    res = patching.causal_corruption_test(
        model, full_tokens, cot_start, cot_end, target_ids_mnli, corrupt_fn
    )
    clean_argmax = res['clean_argmax']
    corrupted_argmax = res['corrupted_argmax']
    patch_records.append({
        'idx': i,
        'label': ex['label'],
        'parsed_answer': parsed,
        'pred_clean': mnli_choices[clean_argmax],
        'pred_corrupted': mnli_choices[corrupted_argmax],
        'flipped': bool(res['flipped']),
        'logit_diff_drop': float(res['logit_diff_drop']),
        'p_clean': res['clean'].tolist(),
        'p_corrupted': res['corrupted'].tolist(),
        'correct_clean': clean_argmax == ex['label'],
        'correct_corrupted': corrupted_argmax == ex['label'],
        'cot_len': cot_end - cot_start,
    })
    if (i + 1) % 25 == 0:
        model_mod.free_memory()

pd.DataFrame(patch_records).to_csv(RESULTS / 'mnli_patching.csv', index=False)
len(patch_records)

In [ ]:
psummary = analysis.patching_summary(patch_records)
print(json.dumps(psummary, indent=2, default=float))

In [ ]:
df = pd.DataFrame(patch_records)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df['logit_diff_drop'], bins=25, ax=axes[0])
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Drop in P(clean answer) after CoT corruption')
axes[0].set_xlabel('clean P(answer) − corrupted P(answer)')
sns.countplot(data=df, x='flipped', ax=axes[1])
axes[1].set_title(f"Flip rate = {df['flipped'].mean():.2%}")
plt.tight_layout(); plt.savefig(RESULTS / 'mnli_corruption.png', dpi=150); plt.show()

## 7. Stub: residual-stream activation patching (post-milestone work)
Below: cache clean pre-CoT residual streams, run the corrupted sequence with the clean pre-CoT slice patched back in, and compare. This isolates whether the corrupted CoT changes the answer *given* that the model entered the CoT span with the same internal state. For the milestone we verify the API runs; the full sweep lands next week.

In [ ]:
if patch_records:
    r = patch_records[0]
    ex = data.mnli_example(mnli[r['idx']])
    prompt = prompts.format_mnli(ex['premise'], ex['hypothesis'])
    prompt_tokens = model.to_tokens(prompt)
    completion, full_tokens = generation.generate_cot(model, prompt, max_new_tokens=160)
    cot_start = prompt_tokens.shape[1]
    corrupted = patching.build_corrupted_tokens(
        full_tokens, cot_start, full_tokens.shape[1],
        lambda x: patching.corrupt_random(model, x, seed=r['idx'])
    )
    patched_probs = patching.patch_pre_cot(
        model, full_tokens, corrupted, cot_start, target_ids_mnli
    )
    print('patched answer distribution:', patched_probs.tolist())

## 8. Save artifacts

In [ ]:
artifact = {
    'n_layers': model.cfg.n_layers,
    'model_name': model.cfg.model_name,
    'tau': TAU,
    'boolq_summary': summary,
    'mnli_patching_summary': psummary,
}
with open(RESULTS / 'milestone_summary.json', 'w') as f:
    json.dump(artifact, f, indent=2, default=float)
print('Saved to', RESULTS / 'milestone_summary.json')